# Практика о квакшах

Достанем текст из Word, PDF и презентации. Затем для каждого формата отдельно очистим строки, оставим русский язык и найдём повторы. Будем сохранять имя файла, место фрагмента и исходную строку, чтобы можно было сверяться с оригиналом.

Запускайте ячейки по порядку из папки `bad_data`.

In [ ]:
# Если нужно установить библиотеки:
# %pip install python-docx python-pptx pypdf langid rapidfuzz pandas -q

from pathlib import Path
from itertools import combinations
import re
import unicodedata
import langid
import pandas as pd
from docx import Document
from pptx import Presentation
from pypdf import PdfReader
from rapidfuzz import fuzz

data_dir = Path('data')

## 1. Word

Соберём сведения об абзацах в датафрейм: номер, стиль, текст и длину.

In [ ]:
document = Document(data_dir / 'frogs_report.docx')
paragraphs = []
for number, paragraph in enumerate(document.paragraphs):
    paragraphs.append({'place': number, 'style': paragraph.style.name,
                       'raw': paragraph.text, 'length': len(paragraph.text)})
paragraphs_df = pd.DataFrame(paragraphs)
paragraphs_df

,place,style,raw,length
0,0,Normal,РЕФЕРАТ ПО БИОЛОГИИ,19
1,1,Title,Квакши и другие\nдревесные лягушки,33
2,2,Subtitle,"Строение, образ жизни и размножение",35
3,3,Normal,,0
4,4,Caption,Квакша Савиньи (Hyla savignyi). Фото: Ester In...,69
...,...,...,...,...
79,79,Normal,Рис. 3: эмбрионы Agalychnis callidryas. Geoff ...,118
80,80,Normal,Рис. 4: Litoria phyllochroa. Froggydarb. CC BY...,107
81,81,Heading 2,Условия использования текста,28
82,82,Normal,Адаптированный текст на основе статей Википеди...,286


Попробуем автоматически выбрать содержательные фрагменты по длине: оставим абзацы от 100 символов. Дополнительно возьмём только обычные абзацы и пункты списка, чтобы убрать заголовки и подписи по их стилю.

Это приближённый отбор. Короткий пункт списка может быть содержательным, а длинная строка с источниками или лицензией может остаться. Поэтому после такого отбора нужно просмотреть таблицу и решить, подходит ли результат для задачи.

In [ ]:
min_length = 100

docx_fragments = []
for paragraph in paragraphs:
    if paragraph['length'] >= min_length and paragraph['style'] in {'Normal', 'List Bullet'}:
        docx_fragments.append({'source': 'frogs_report.docx',
                               'place': paragraph['place'], 'raw': paragraph['raw']})
pd.DataFrame(docx_fragments)

,source,place,raw
0,frogs_report.docx,9,"Квакши известны как небольшие лягушки, которые..."
1,frogs_report.docx,10,Название Hylidae обозначает семейство в отряде...
2,frogs_report.docx,12,В центре этого реферата находятся квакши и нес...
3,frogs_report.docx,13,Квакши живут среди растений. Расширенные подуш...
4,frogs_report.docx,23,Одна из самых заметных особенностей многих дре...
5,frogs_report.docx,24,Окраска представителей этой группы различается...
6,frogs_report.docx,25,Квакшы жывуць сярод раслін. Пашыраныя падушачк...
7,frogs_report.docx,28,Голос особенно важен в период размножения. У с...
8,frogs_report.docx,29,У многих видов головастики развиваются в воде....
9,frogs_report.docx,32,Обыкновенная квакша (Hyla arborea) - небольшая...


Проверим таблицы отдельно. Они не входят в `document.paragraphs`. Колонтитулы, сноски и текстовая врезка тоже остались за пределами этого списка. Для основного решения их не добавляем.

In [ ]:
print('Таблиц:', len(document.tables))
for table in document.tables:
    for row in table.rows:
        print([cell.text for cell in row.cells])

Таблиц: 3
['Уровень классификации', 'Русское название', 'Латинское название']
['Класс', 'Земноводные', 'Amphibia']
['Отряд', 'Бесхвостые', 'Anura']
['Семейство', 'Квакши', 'Hylidae']
['Вид', 'Регион', 'Размер и заметные признаки']
['Обыкновенная квакша\nHyla arborea', 'Европа', 'Около 5\xa0см; тёмная боковая полоса']
['Красноглазая квакша\nAgalychnis callidryas', 'Юг Мексики и Центральная Америка', 'Самки до 7,7\xa0см, самцы до 5,9\xa0см; красные глаза']
['Термин', 'Значение']
['Ареал', 'Область распространения вида']
['Головастик', 'Личиночная стадия бесхвостой амфибии']
['Метаморфоз', 'Перестройка организма при переходе к взрослой форме']
['Адгезия', 'Сцепление соприкасающихся поверхностей']


## 2. PDF

Достанем текст всех страниц и соберём датафрейм: номер страницы, текст и длина. Страницы нумеруются с нуля.

In [ ]:
reader = PdfReader(data_dir / 'frogs_report.pdf')
pdf_pages = []
for page_number, page in enumerate(reader.pages):
    text = page.extract_text()
    pdf_pages.append({'source': 'frogs_report.pdf', 'place': page_number,
                      'raw': text, 'length': len(text)})
pd.DataFrame(pdf_pages)

,source,place,raw,length
0,frogs_report.pdf,0,Реферат по биологии • 2026\nРЕФЕРАТ ПО БИОЛО...,337
1,frogs_report.pdf,1,КВАКШИ / БИОЛОГИЯ ЗЕМНОВОДНЫХ\nКвакши и друг...,1980
2,frogs_report.pdf,2,КВАКШИ / БИОЛОГИЯ ЗЕМНОВОДНЫХ\nКвакши и друг...,1786
3,frogs_report.pdf,3,КВАКШИ / БИОЛОГИЯ ЗЕМНОВОДНЫХ\nКвакши и друг...,1630
4,frogs_report.pdf,4,КВАКШИ / БИОЛОГИЯ ЗЕМНОВОДНЫХ\nКвакши и друг...,1747
5,frogs_report.pdf,5,КВАКШИ / БИОЛОГИЯ ЗЕМНОВОДНЫХ\nКвакши и друг...,2086
6,frogs_report.pdf,6,КВАКШИ / БИОЛОГИЯ ЗЕМНОВОДНЫХ\nКвакши и друг...,1332


В PDF нет готовых абзацев, как в Word. Попробуем простое правило: копим строки, пока строка не заканчивается точкой, вопросительным или восклицательным знаком. Учитываем номер сноски после точки.

Посмотрим на первые строки, которыми заканчивались бы фрагменты. В регулярном выражении `[.!?]` - нужный знак, `(\s*\d+)?` - необязательные пробелы и номер сноски, `$` - конец строки.

In [ ]:
endings = []
for page in pdf_pages:
    for line in page['raw'].splitlines():
        line = line.strip()
        if re.search(r'[.!?](\s*\d+)?$', line):
            endings.append({'page': page['place'], 'line': line})
pd.DataFrame(endings).head(10)

,page,line
0,0,Квакша Савиньи (Hyla savignyi). Фото: Ester In...
1,0,"Почему лягушка может жить на ветке, но её разв..."
2,0,По материалам Википедии и исследований биологи...
3,0,Ноябрь 2026 года.
4,1,стадия.
5,1,сравнения источников. 1
6,1,Таблица 1. Место семейства Hylidae среди позво...
7,1,русские названия похожи.
8,1,на листьях и ветвях.
9,1,намеренно не фиксируется; для систематики след...


Теперь соберём фрагменты для всех страниц. Две строки колонтитулов пропустим по их началу. Остаток в конце страницы тоже сохраняем, даже если у него нет точки. В поле `place` запишем пару «страница, фрагмент».

Это приближённое восстановление: заголовок может склеиться с абзацем, а абзац иногда разделится на части. Для наших вставок правило сработало. После обработки всё равно надо сравнить вывод с документом, особенно если есть колонки.

In [ ]:
pdf_all_fragments = []
for page in pdf_pages:
    current = []
    fragment_number = 0
    for line in page['raw'].splitlines():
        line = line.strip()
        if not line or line.startswith('КВАКШИ'):
            continue
        if re.match(r'Квакши и другие древесные лягушки \d', line):
            continue
        current.append(line)
        if re.search(r'[.!?](\s*\d+)?$', line):
            text = ' '.join(current)
            pdf_all_fragments.append({'source': 'frogs_report.pdf',
                'place': (page['place'], fragment_number), 'raw': text, 'length': len(text)})
            fragment_number += 1
            current = []
    if current:
        text = ' '.join(current)
        pdf_all_fragments.append({'source': 'frogs_report.pdf',
            'place': (page['place'], fragment_number), 'raw': text, 'length': len(text)})
pd.DataFrame(pdf_all_fragments)

,source,place,raw,length
0,frogs_report.pdf,"(0, 0)",Реферат по биологии • 2026 РЕФЕРАТ ПО БИОЛОГ...,188
1,frogs_report.pdf,"(0, 1)","Почему лягушка может жить на ветке, но её разв...",73
2,frogs_report.pdf,"(0, 2)",По материалам Википедии и исследований биологи...,56
3,frogs_report.pdf,"(0, 3)",Ноябрь 2026 года.,17
4,frogs_report.pdf,"(1, 0)",Квакши среди земноводных Квакши известны как н...,449
5,frogs_report.pdf,"(1, 1)",Название Hylidae обозначает семейство в отряде...,362
6,frogs_report.pdf,"(1, 2)",Уровень классификации Русское название Латинск...,186
7,frogs_report.pdf,"(1, 3)",В центре этого реферата находятся квакши и нес...,358
8,frogs_report.pdf,"(1, 4)",Квакши живут среди растений. Расширенные подуш...,104
9,frogs_report.pdf,"(1, 5)",Разделы реферата Строение и голос Два вида для...,314


Как и в Word, оставим фрагменты от 100 символов.

In [ ]:
pdf_fragments = [fragment for fragment in pdf_all_fragments
                 if fragment['length'] >= min_length]
pd.DataFrame(pdf_fragments)

,source,place,raw,length
0,frogs_report.pdf,"(0, 0)",Реферат по биологии • 2026 РЕФЕРАТ ПО БИОЛОГ...,188
1,frogs_report.pdf,"(1, 0)",Квакши среди земноводных Квакши известны как н...,449
2,frogs_report.pdf,"(1, 1)",Название Hylidae обозначает семейство в отряде...,362
3,frogs_report.pdf,"(1, 2)",Уровень классификации Русское название Латинск...,186
4,frogs_report.pdf,"(1, 3)",В центре этого реферата находятся квакши и нес...,358
5,frogs_report.pdf,"(1, 4)",Квакши живут среди растений. Расширенные подуш...,104
6,frogs_report.pdf,"(1, 5)",Разделы реферата Строение и голос Два вида для...,314
7,frogs_report.pdf,"(1, 6)",Древесный образ жизни объединяет лягушек по сп...,113
8,frogs_report.pdf,"(2, 0)",Строение и голос Одна из самых заметных особен...,384
9,frogs_report.pdf,"(2, 1)",Окраска представителей этой группы различается...,341


В таком выводе могут остаться подписи, сноски и части таблиц. Мы не восстановили структуру PDF полностью.

## 3. Презентация

Обойдём все слайды и соберём текстовые блоки в датафрейм. У каждого блока сохраним пару «номер слайда, номер объекта», имя, текст и длину.

In [ ]:
presentation = Presentation(data_dir / 'frogs_slides.pptx')
pptx_all_fragments = []
for slide_number, slide in enumerate(presentation.slides):
    for shape_number, shape in enumerate(slide.shapes):
        if shape.has_text_frame and shape.text.strip():
            pptx_all_fragments.append({'source': 'frogs_slides.pptx',
                'place': (slide_number, shape_number), 'name': shape.name,
                'raw': shape.text, 'length': len(shape.text)})
pd.DataFrame(pptx_all_fragments)

,source,place,name,raw,length
0,frogs_slides.pptx,"(0, 0)",cover-subject,РЕФЕРАТ ПО БИОЛОГИИ,19
1,frogs_slides.pptx,"(0, 1)",cover-title,Квакши и другие\nдревесные лягушки,33
2,frogs_slides.pptx,"(0, 2)",cover-subtitle,"Строение, образ жизни\nи размножение",35
3,frogs_slides.pptx,"(0, 3)",cover-date,Ноябрь 2026 года,16
4,frogs_slides.pptx,"(0, 5)",photo-caption-hyla,"Hyla savignyi. Фото: Ester Inbar, Attribution.",46
5,frogs_slides.pptx,"(1, 0)",slide-title,Квакши среди земноводных,24
6,frogs_slides.pptx,"(1, 1)",footer,Квакши и другие древесные лягушки,33
7,frogs_slides.pptx,"(1, 2)",page-number,2,1
8,frogs_slides.pptx,"(1, 4)",taxonomy-explanation,«Древесные лягушки» описывает образ жизни.\n\n...,95
9,frogs_slides.pptx,"(1, 5)",toe-pad-explanation,Расширенные подушечки пальцев помогают удержив...,111


Оставим блоки от 100 символов. По именам объектов дополнительно уберём колонтитулы, заголовки и подписи к фотографиям.

In [ ]:
pptx_fragments = []
for fragment in pptx_all_fragments:
    if fragment['length'] >= min_length:
        if fragment['name'] not in {'slide-title', 'footer', 'page-number'}:
            if not fragment['name'].startswith('photo-caption'):
                pptx_fragments.append(fragment)
pd.DataFrame(pptx_fragments)

,source,place,name,raw,length
0,frogs_slides.pptx,"(1, 5)",toe-pad-explanation,Расширенные подушечки пальцев помогают удержив...,111
1,frogs_slides.pptx,"(1, 6)",observation-1,Квакши живут среди растений. Расширенные подуш...,104
2,frogs_slides.pptx,"(1, 7)",observation-8,Обыкновенная квакша (Hyla arborea) относится к...,142
3,frogs_slides.pptx,"(2, 4)",calling-function,Сигналы помогают привлекать партнёра и взаимод...,116
4,frogs_slides.pptx,"(2, 7)",observation-4,Квакшы жывуць сярод раслін. Пашыраныя падушачк...,146
5,frogs_slides.pptx,"(3, 7)",species-description,"Красные глаза, зелёная спина и яркие пальцы.\n...",131
6,frogs_slides.pptx,"(3, 8)",observation-5,Квакши живут среди растений. Расширенные подуш...,104
7,frogs_slides.pptx,"(4, 4)",embryo-explanation,На фотографии видны эмбрионы Agalychnis callid...,159
8,frogs_slides.pptx,"(4, 7)",observation-2,У многих видов головастики развиваются в воде....,119
9,frogs_slides.pptx,"(5, 9)",observation-6,У многих видов головастики развиваются в воде....,119


Таблицы и заметки докладчика читаются отдельно. Обойдём все слайды и сложим в таблицу строки таблиц и заметки. В основной список их не добавляем, чтобы не смешивать текст на слайде с его развёрнутым пересказом.

In [ ]:
extra_fragments = []
for slide_number, slide in enumerate(presentation.slides):
    for shape_number, shape in enumerate(slide.shapes):
        if shape.has_table:
            for row_number, row in enumerate(shape.table.rows):
                extra_fragments.append({'place': (slide_number, shape_number, row_number),
                    'type': 'table_row', 'text': ' | '.join(cell.text for cell in row.cells)})
    if slide.has_notes_slide:
        extra_fragments.append({'place': (slide_number,), 'type': 'notes',
                                'text': slide.notes_slide.notes_text_frame.text})
pd.DataFrame(extra_fragments)

,place,type,text
0,"(0,)",notes,"Квакши известны как небольшие лягушки, которые..."
1,"(1, 3, 0)",table_row,Уровень | Название
2,"(1, 3, 1)",table_row,Класс | Земноводные
3,"(1, 3, 2)",table_row,Отряд | Бесхвостые
4,"(1, 3, 3)",table_row,Семейство | Hylidae
5,"(1,)",notes,Название Hylidae обозначает семейство в отряде...
6,"(2,)",notes,Голос особенно важен в период размножения. У с...
7,"(3,)",notes,Красноглазая квакша (Agalychnis callidryas) об...
8,"(4,)",notes,Связь с водой особенно заметна при размножении...
9,"(5,)",notes,Из яйца у многих бесхвостых выходит головастик...


## 4. Очищаем извлечённые тексты

Для трёх наборов нужны одни и те же шаги. Сложим их в словарь, но каждый набор будем обрабатывать отдельно.

Внутри одного фрагмента заменим любые пробельные символы обычным пробелом. В отличие от основной тетрадки, здесь можно убрать переводы строк: отдельные фрагменты уже хранятся в отдельных словарях.

In [ ]:
def clean_text(text):
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\u00ad', '')
    return re.sub(r'\s+', ' ', text).strip()

collections = {'DOCX': docx_fragments, 'PDF': pdf_fragments, 'PPTX': pptx_fragments}
for name, fragments in collections.items():
    for fragment in fragments:
        fragment['text'] = clean_text(fragment['raw'])
    print(name, '-', len(fragments), 'фрагментов')

DOCX - 29 фрагментов
PDF - 39 фрагментов
PPTX - 14 фрагментов


Посмотрим, что изменилось у абзаца о головастиках. `repr()` позволит увидеть разные пробелы в исходной строке.

In [ ]:
for name, fragments in collections.items():
    print(name)
    for fragment in fragments:
        if fragment['text'].startswith('У многих видов головастики'):
            print(fragment['place'], repr(fragment['raw']))
            print(fragment['text'], end='\n\n')

DOCX
29 'У многих видов головастики развиваются в воде. Во время метаморфоза появляются конечности, а хвост постепенно исчезает.'
У многих видов головастики развиваются в воде. Во время метаморфоза появляются конечности, а хвост постепенно исчезает.

46 'У\xa0многих видов головастики развиваются в воде. Во\xa0время метаморфоза появляются\u202fконечности, а хвост постепенно исчезает.'
У многих видов головастики развиваются в воде. Во время метаморфоза появляются конечности, а хвост постепенно исчезает.

PDF
(2, 6) 'У многих видов головастики развиваются в воде. Во время метаморфоза появляются конечности, а хвост постепенно исчезает.'
У многих видов головастики развиваются в воде. Во время метаморфоза появляются конечности, а хвост постепенно исчезает.

(4, 5) 'У\xa0многих видов головастики развиваются в воде. Во\xa0время метаморфоза появляются\u202fконечности, а хвост постепенно исчезает.'
У многих видов головастики развиваются в воде. Во время метаморфоза появляются конечности, а хвост

## 5. Проверяем язык

Сначала посмотрим на всё, что модель не назвала русским. Не будем автоматически удалять короткие строки: как мы видели на паре, на них особенно легко ошибиться.

In [ ]:
for name, fragments in collections.items():
    print(name)
    for fragment in fragments:
        language, score = langid.classify(fragment['text'])
        fragment['language'] = language
        if language != 'ru':
            print(fragment['place'], language, '-', fragment['text'], end='\n\n')

DOCX
25 be - Квакшы жывуць сярод раслін. Пашыраныя падушачкі пальцаў дапамагаюць ім трымацца на лістах і галінах. У многіх відаў апалонікі развіваюцца ў вадзе.

50 en - Tree frogs live among plants. Their expanded toe pads help them hold on to leaves and branches. In many species, tadpoles develop in water.

65 en - Tree frogs illustrate the connection between body structure, behaviour and habitat. Expanded toe pads support climbing, while vocal signals are important during breeding. In many species, the life cycle includes an aquatic larval stage. Common names describe lifestyle and do not always correspond to a single taxonomic group.

PDF
(2, 2) be - Квакшы жывуць сярод раслін. Пашыраныя падушачкі пальцаў дапамагаюць ім трымацца на лістах і галінах. У многіх відаў апалонікі развіваюцца ў вадзе.

(4, 9) en - Tree frogs live among plants. Their expanded toe pads help them hold on to leaves and branches. In many species, tadpoles develop in water.

(5, 8) en - Summary Tree frogs illust

Посмотрим на предсказания выше и оставим фрагменты, для которых модель вернула `ru`. Белорусские и английские вставки исключим. Короткие строки, на которых модель особенно часто ошибается, уже не прошли отбор по длине.

In [ ]:
russian = {}
for name, fragments in collections.items():
    russian[name] = [f for f in fragments if f['language'] == 'ru']
    print(name, '- оставили', len(russian[name]), 'фрагментов')

DOCX - оставили 26 фрагментов
PDF - оставили 36 фрагментов
PPTX - оставили 12 фрагментов


Проверим, что научное название не потерялось вместе с английскими вставками.

In [ ]:
for name, fragments in russian.items():
    print(name)
    for fragment in fragments:
        if 'Hyla arborea' in fragment['text']:
            print(fragment['place'], '-', fragment['text'])

DOCX
32 - Обыкновенная квакша (Hyla arborea) - небольшая европейская лягушка. В статье о ней указана длина тела около 5 см. Характерный признак - тёмная полоса по бокам головы и туловища. Животное может быть зелёным, сероватым или бурым; окраска меняется. Для размножения подходят хорошо прогреваемые стоячие водоёмы с растительностью. Икру самка откладывает небольшими порциями, прикрепляя кладки к растениям или размещая их на дне.
63 - Обыкновенная квакша (Hyla arborea) относится к земноводным. Латинское название помогает различать виды, даже когда их русские названия похожи.
PDF
(3, 0) - Два вида для сравнения Обыкновенная квакша (Hyla arborea) - небольшая европейская лягушка. В статье о ней указана длина тела около 5 см. Характерный признак - тёмная полоса по бокам головы и туловища. Животное может быть зелёным, сероватым или бурым; окраска меняется. Для размножения подходят хорошо прогреваемые стоячие водоёмы с растительностью. Икру самка откладывает небольшими порциями, прикрепляя к

## 6. Находим повторы

Сначала используем `dict.fromkeys()`, как на паре. Он оставит первое появление каждой строки. Через `.index()` найдём её первый номер в списке и сохраним соответствующий словарь с источником.

Посмотрим на убранные повторы. Две копии абзаца о головастиках совпадут только после очистки пробелов.

In [ ]:
exact = {}
for name, fragments in russian.items():
    texts = [f['text'] for f in fragments]
    unique_texts = list(dict.fromkeys(texts))
    first_indices = [texts.index(text) for text in unique_texts]
    exact[name] = [fragments[i] for i in first_indices]
    print(name)
    for i, fragment in enumerate(fragments):
        if i not in first_indices:
            first = fragments[texts.index(fragment['text'])]
            print('Оставили', first['place'], '| убрали', fragment['place'])
            print(repr(first['raw']))
            print(repr(fragment['raw']), end='\n\n')

DOCX
Оставили 13 | убрали 37
'Квакши живут среди растений. Расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.'
'Квакши живут среди растений. Расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.'

Оставили 29 | убрали 46
'У многих видов головастики развиваются в воде. Во время метаморфоза появляются конечности, а хвост постепенно исчезает.'
'У\xa0многих видов головастики развиваются в воде. Во\xa0время метаморфоза появляются\u202fконечности, а хвост постепенно исчезает.'

PDF
Оставили (1, 4) | убрали (3, 4)
'Квакши живут среди растений. Расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.'
'Квакши живут среди растений. Расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.'

Оставили (2, 6) | убрали (4, 5)
'У многих видов головастики развиваются в воде. Во время метаморфоза появляются конечности, а хвост постепенно исчезает.'
'У\xa0многих видов головастики развиваются в воде. Во\xa0время метаморф

Теперь посмотрим на пары со сходством от 80. Это пока только кандидаты: высокая оценка сама по себе не доказывает, что один текст можно удалить.

In [ ]:
for name, fragments in exact.items():
    print(name)
    for first, second in combinations(fragments, 2):
        score = fuzz.ratio(first['text'], second['text'])
        if score >= 80:
            print(round(score, 2), '-', first['place'], '/', second['place'])
            print(first['text'])
            print(second['text'], end='\n\n')

DOCX
98.08 - 13 / 56
Квакши живут среди растений. Расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.
Квакши живут среди растений: расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.

94.29 - 77 / 80
Рис. 1: Litoria chloris. Froggydarb. CC BY-SA 3.0. Страница файла на Wikimedia Commons Условия лицензии
Рис. 4: Litoria phyllochroa. Froggydarb. CC BY-SA 3.0. Страница файла на Wikimedia Commons Условия лицензии

PDF
98.08 - (1, 4) / (5, 2)
Квакши живут среди растений. Расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.
Квакши живут среди растений: расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.

PPTX
98.08 - (1, 6) / (6, 5)
Квакши живут среди растений. Расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.
Квакши живут среди растений: расширенные подушечки пальцев помогают им удерживаться на листьях и ветвях.



У повторного абзаца о подушечках точка заменена двоеточием, а прописная буква стала строчной. Сходство около 98.08; смысл не изменился. Для этих данных выберем порог 97 и оставим первую версию.

В другой коллекции порог придётся выбирать заново после просмотра пар.

In [ ]:
results = {}
for name, fragments in exact.items():
    result = []
    print(name)
    for fragment in fragments:
        duplicate = False
        for previous in result:
            score = fuzz.ratio(previous['text'], fragment['text'])
            if score >= 97:
                duplicate = True
                print('Убрали', fragment['place'], '- сходство', round(score, 2))
                break
        if not duplicate:
            result.append(fragment)
    results[name] = result

DOCX
Убрали 56 - сходство 98.08
PDF
Убрали (5, 2) - сходство 98.08
PPTX
Убрали (6, 5) - сходство 98.08


## 7. Проверяем результат

Сравним число фрагментов и символов на каждом шаге.

In [ ]:
for name, fragments in collections.items():
    print(name)
    print('Исходный текст:', sum(len(f['raw']) for f in fragments), 'символов')
    print('После очистки:', sum(len(f['text']) for f in fragments), 'символов')
    print('Фрагменты:', len(fragments), '-> русский язык:', len(russian[name]),
          '-> без точных повторов:', len(exact[name]), '-> без похожих повторов:', len(results[name]))
    print('В результате:', sum(len(f['text']) for f in results[name]), 'символов', end='\n\n')

DOCX
Исходный текст: 7182 символов
После очистки: 7177 символов
Фрагменты: 29 -> русский язык: 26 -> без точных повторов: 24 -> без похожих повторов: 23
В результате: 6239 символов

PDF
Исходный текст: 9602 символов
После очистки: 9600 символов
Фрагменты: 39 -> русский язык: 36 -> без точных повторов: 34 -> без похожих повторов: 33
В результате: 8654 символов

PPTX
Исходный текст: 1843 символов
После очистки: 1838 символов
Фрагменты: 14 -> русский язык: 12 -> без точных повторов: 10 -> без похожих повторов: 9
В результате: 1226 символов



Посмотрим на первые фрагменты каждого результата. Полный список хранится в `results['DOCX']`, `results['PDF']` и `results['PPTX']`. Исходные строки и номера мест по-прежнему доступны в каждой записи.

In [ ]:
for name, fragments in results.items():
    print(name)
    for fragment in fragments[:3]:
        print(fragment['place'], '-', fragment['text'], end='\n\n')

DOCX
9 - Квакши известны как небольшие лягушки, которые держатся на листьях и ветвях. Их пример показывает, как строение тела связано с образом жизни. Расширенные кончики пальцев помогают удерживаться на поверхности, длинные задние конечности позволяют прыгать, а окраска делает животное менее заметным среди растений. При этом древесный образ жизни не устраняет связи с водой: у многих видов именно в ней проходит личиночная стадия.

10 - Название Hylidae обозначает семейство в отряде бесхвостых земноводных. Однако бытовое выражение «древесные лягушки» прежде всего описывает образ жизни. Границы семейств и положение отдельных родов пересматриваются: разные справочники могут использовать разные схемы. Поэтому число видов без даты и указания классификации мало подходит для сравнения источников.

12 - В центре этого реферата находятся квакши и несколько близких по образу жизни древесных лягушек. Рассматриваются их место среди земноводных, приспособления к жизни на растительности, голосовое п